## 1 — List input files

In [1]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU count: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [2]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/ahmedzayed2/working-laws-json/labor_law_14_2025.json
/kaggle/input/datasets/ahmedzayed2/working-laws-json/issuing_law_14_2025.json


## 2 — Install packages

In [3]:
!pip install faiss-cpu langchain-community langchain-huggingface langchain-classic sentence-transformers transformers==4.52.4 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 112.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 90.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 98.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not in

## 3 — Imports

In [4]:
import json
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from transformers import AutoModelForCausalLM, AutoTokenizer

print('All imports successful.')

/tmp/ipykernel_58/2117755688.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


All imports successful.


## 4 — Add effective_date and save to /kaggle/working/

In [5]:
labor_law_path_inp   = '/kaggle/input/datasets/ahmedzayed2/working-laws-json/labor_law_14_2025.json'
issuing_law_path_inp = '/kaggle/input/datasets/ahmedzayed2/working-laws-json/issuing_law_14_2025.json'
labor_law   = '/kaggle/working/labor_law_14_2025.json'
issuing_law = '/kaggle/working/issuing_law_14_2025.json'

for inp, out in [(labor_law_path_inp, labor_law), (issuing_law_path_inp, issuing_law)]:
    with open(inp, 'r', encoding='utf-8') as f:
        data = json.load(f)
    for article in data:
        article['effective_date'] = '2025-08-01'
    with open(out, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f'Saved {len(data)} articles → {out}')

# quick check
with open(labor_law, 'r', encoding='utf-8') as f:
    check = json.load(f)
print(f'effective_date present: {"effective_date" in check[0]}')

Saved 299 articles → /kaggle/working/labor_law_14_2025.json
Saved 13 articles → /kaggle/working/issuing_law_14_2025.json
effective_date present: True


## 5 — Build Documents and FAISS Index

In [6]:
# load both JSON files
with open(labor_law, 'r', encoding='utf-8') as f:
    mwafaq = json.load(f)
with open(issuing_law, 'r', encoding='utf-8') as f:
    isdar = json.load(f)

all_articles = mwafaq + isdar
print(f'Total articles: {len(all_articles)}')

# convert to LangChain Documents
documents = []
for article in all_articles:
    content = ''
    if article.get('book'):
        content += f"{article['book']} - "
    if article.get('chapter'):
        content += f"{article['chapter']} - "
    content += article['text']

    doc = Document(
        page_content=content,
        metadata={
            'document':      article['document'],
            'article':       article['article'],
            'book':          article.get('book'),
            'chapter':       article.get('chapter'),
            'article_label': article.get('article_label'),
            'source':        article['source'],
            'status':        article['status'],
        }
    )
    documents.append(doc)

print(f'Documents to index: {len(documents)}')
print(f'\nSample document content:')
print(documents[0].page_content[:200])

Total articles: 312
Documents to index: 312

Sample document content:
التعاريف والأحكام العامة - التعاريف - يقصد فى تطبيق أحكام هذا القانون بالكلمات والمصطلحات الآتية المعانى المبينة قرين كل منها : -١ العامل : كل شخص طبيعى يعمل لقاء أجر لدى صاحب عمل تحت إدارتـه أو إشراف


## 6 — Embed and Save Index

In [7]:
embedding_model = HuggingFaceEmbeddings(
    model_name='sentence-transformers/paraphrase-multilingual-mpnet-base-v2',
    model_kwargs={'device': 'cpu'},  # CPU only — avoids CUDA version conflict
    encode_kwargs={'normalize_embeddings': True}
)

vectordb = FAISS.from_documents(documents, embedding_model)
vectordb.save_local('legal_index')
print(f'Index saved. Total vectors: {vectordb.index.ntotal}')

# verify scores are now in 0.0 - 1.0 range
test = vectordb.similarity_search_with_score('ما هو تعريف العامل؟', k=3)
print('\nTest scores (should be 0.0 - 1.0):')
for doc, score in test:
    print(f'  Score {score:.3f} → المادة ({doc.metadata["article"]})')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Index saved. Total vectors: 312

Test scores (should be 0.0 - 1.0):
  Score 0.616 → المادة (69)
  Score 0.622 → المادة (1)
  Score 0.814 → المادة (96)


## 7 — Load LLM

In [8]:
model_name = 'mistralai/Mistral-Nemo-Instruct-2407'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map='auto'  # automatically splits across both T4s
)

# confirm which devices are being used
print('Model loaded.')
print(f'Devices used: {set(str(p.device) for p in model.parameters())}')

def generate_answer(prompt, max_new_tokens=500):
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda:0')
    input_length = inputs['input_ids'].shape[1]

    if input_length > 3000:
        print(f'Warning: prompt is {input_length} tokens')

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.1,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )
    new_tokens = outputs[0][input_length:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded.
Devices used: {'cuda:1', 'cuda:0'}


## 8 — Retrieve Function

In [9]:
def retrieve(question, k=7):
    results = vectordb.similarity_search_with_score(question, k=k)
    results.sort(key=lambda x: x[1])

    seen = set()
    unique = []
    for doc, score in results:
        key = (doc.metadata['document'], doc.metadata['article'])
        if key not in seen:
            seen.add(key)
            unique.append((doc, score))

    THRESHOLD = 1.3
    filtered = [(doc, score) for doc, score in unique if score < THRESHOLD]

    if not filtered:
        filtered = [unique[0]]
        print(f'Warning: best score {unique[0][1]:.3f}')

    print(f'Retrieved {len(filtered)} articles:')
    for doc, score in filtered:
        print(f'  Score {score:.3f} → المادة ({doc.metadata["article"]}) | {doc.metadata.get("document","")} | {doc.metadata.get("book","")}')

    return [doc for doc, score in filtered]

## 9 — Build Prompt Function

In [10]:
def build_prompt(question, docs):
    context_parts = []
    for doc in docs:
        meta = doc.metadata
        label = f"المادة {meta['article_label']}" if meta.get('article_label') else f"المادة ({meta['article']})"
        book    = f" | {meta['book']}"    if meta.get('book')    else ''
        chapter = f" | {meta['chapter']}" if meta.get('chapter') else ''
        context_parts.append(f"{label}{book}{chapter}:\n{doc.page_content}")

    context = '\n\n---\n\n'.join(context_parts)

    return f"""<s>[INST] أنت مساعد قانوني متخصص في قانون العمل المصري رقم 14 لسنة 2025.

مهمتك:
- اقرأ النصوص القانونية وأجب على السؤال منها مباشرة.
- اذكر رقم المادة التي استندت إليها.
- إذا كانت النصوص تتعلق بالسؤال ولو بشكل غير مباشر، استخلص منها الإجابة.
- إذا لم تكن النصوص تجيب على السؤال بشكل مباشر، قدم ما هو أقرب إجابة ممكنة من النصوص المتاحة وأوضح ذلك للمستخدم.
- لا تخترع معلومات غير موجودة في النصوص.
- لا تقل أبدًا "لا توجد معلومات" إلا إذا كانت النصوص لا علاقة لها بالسؤال تمامًا.

النصوص القانونية:
{context}

السؤال: {question} [/INST]"""

## 10 — Full Pipeline

In [11]:
def ask_legal_question(question):
    print(f'\nالسؤال: {question}')
    print('='*50)

    docs = retrieve(question, k=7)

    if not docs:
        return 'لا توجد معلومات كافية في القانون المتاح'

    prompt = build_prompt(question, docs)
    return generate_answer(prompt)

## 11 — Test

In [12]:
normal_questions = [
    'ما هي مدة الإجازة السنوية للعامل؟',
    'ما هي حقوق العامل عند الفصل التعسفي؟',
    'كم ساعة العمل اليومي المسموح به؟',
    'ما هو تعريف العامل في القانون؟',
    'متى بدأ العمل بهذا القانون؟',
    'ما هي شروط عقد العمل؟',
    'ما هو الحد الأدنى للأجر؟',
    
]

edge_cases = [
    'ما هو راتب الرئيس؟',
    'الإجازة؟',
    'ما هى مدة الإجاذة السنوية؟',
]

print('=== Normal Questions ===')
for q in normal_questions:
    answer = ask_legal_question(q)
    print(f'الجواب: {answer}')
    print('-'*60)

print('\n=== Edge Cases ===')
for q in edge_cases:
    answer = ask_legal_question(q)
    print(f'الجواب: {answer}')
    print('-'*60)


import os

print("Files ready for saving:")
for item in os.listdir('/kaggle/working'):
    path = f'/kaggle/working/{item}'
    if os.path.isdir(path):
        total = sum(
            os.path.getsize(os.path.join(path, f))
            for f in os.listdir(path)
        )
        print(f"  📁 {item}/ — {total/1024/1024:.1f} MB")
    else:
        size = os.path.getsize(path)
        print(f"  📄 {item} — {size/1024:.1f} KB")

print("\nClick 'Save Version' → 'Save & Run All' to preserve these files.")

=== Normal Questions ===

السؤال: ما هي مدة الإجازة السنوية للعامل؟
Retrieved 7 articles:
  Score 0.593 → المادة (128) | القانون المرافق | علاقات العمل الفردية
  Score 0.655 → المادة (126) | القانون المرافق | علاقات العمل الفردية
  Score 0.696 → المادة (159) | القانون المرافق | علاقات العمل الفردية
  Score 0.729 → المادة (124) | القانون المرافق | علاقات العمل الفردية
  Score 0.733 → المادة (166) | القانون المرافق | علاقات العمل الفردية
  Score 0.774 → المادة (130) | القانون المرافق | علاقات العمل الفردية
  Score 0.790 → المادة (120) | القانون المرافق | علاقات العمل الفردية
الجواب: مدة الإجازة السنوية للعامل في مصر حسب قانون العمل رقم 14 لسنة 2025 تنقسم إلى عدة فترات حسب مدة خدمة العامل لدى صاحب العمل، حيث:

* في السنة الأولى: 15 يومًا.
* من السنة الثانية وما بعدها: 21 يومًا.
* لمن أمضى 10 سنوات كاملة لدى صاحب عمل أو أكثر، أو لمن تجاوزت سنه 50 عامًا: 30 يومًا.
* للأشخاص ذوي الإعاقة والأقزام: 45 يومًا.

كما يتم زيادة مدة الإجازة السنوية بمقدار 7 أيام للعمال الذين يعملون في الأعمال الخطرة

## 12 — Deployment

In [13]:

import os

print("Files ready for saving:")
for item in os.listdir('/kaggle/working'):
    path = f'/kaggle/working/{item}'
    if os.path.isdir(path):
        # show folder size
        total = sum(
            os.path.getsize(os.path.join(path, f))
            for f in os.listdir(path)
        )
        print(f"  📁 {item}/ — {total/1024/1024:.1f} MB")
    else:
        size = os.path.getsize(path)
        print(f"  📄 {item} — {size/1024:.1f} KB")



Files ready for saving:
  📁 legal_index/ — 1.3 MB
  📁 .virtual_documents/ — 0.0 MB
  📄 labor_law_14_2025.json — 392.6 KB
  📄 issuing_law_14_2025.json — 13.5 KB

Click 'Save Version' → 'Save & Run All' to preserve these files.


In [14]:
app_code = """
import streamlit as st
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

st.set_page_config(
    page_title="المساعد القانوني المصري",
    page_icon="⚖️",
    layout="centered"
)

st.markdown('''
    <style>
    .main { direction: rtl; text-align: right; }
    .stTextArea textarea { direction: rtl; text-align: right; }
    </style>
''', unsafe_allow_html=True)

@st.cache_resource
def load_all():
    embedding_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )
    vectordb = FAISS.load_local(
        "/kaggle/working/legal_index",
        embedding_model,
        allow_dangerous_deserialization=True
    )
    tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-Nemo-Instruct-2407")
    model = AutoModelForCausalLM.from_pretrained(
        "mistralai/Mistral-Nemo-Instruct-2407",
        torch_dtype=torch.float16,
        device_map="auto"
    )
    return vectordb, tokenizer, model

vectordb, tokenizer, model = load_all()

def retrieve(question, k=7):
    results = vectordb.similarity_search_with_score(question, k=k)
    results.sort(key=lambda x: x[1])
    seen = set()
    unique = []
    for doc, score in results:
        key = (doc.metadata["document"], doc.metadata["article"])
        if key not in seen:
            seen.add(key)
            unique.append((doc, score))
    THRESHOLD = 1.3
    filtered = [(doc, score) for doc, score in unique if score < THRESHOLD]
    if not filtered:
        filtered = [unique[0]]
    return filtered

def build_prompt(question, docs):
    context_parts = []
    for doc in docs:
        meta = doc.metadata
        label = f"المادة {meta['article_label']}" if meta.get("article_label") else f"المادة ({meta['article']})"
        book    = f" | {meta['book']}"    if meta.get("book")    else ""
        chapter = f" | {meta['chapter']}" if meta.get("chapter") else ""
        context_parts.append(f"{label}{book}{chapter}:\\n{doc.page_content}")
    context = "\\n\\n---\\n\\n".join(context_parts)
    return f"<s>[INST] أنت مساعد قانوني متخصص في قانون العمل المصري رقم 14 لسنة 2025.\\n\\nمهمتك:\\n- اقرأ النصوص القانونية جيدًا وأجب على السؤال منها مباشرة.\\n- اذكر رقم المادة التي استندت إليها.\\n- إذا كانت النصوص تتعلق بالسؤال ولو بشكل غير مباشر، استخدمها للإجابة.\\n- حاول دائمًا استخلاص الإجابة من النصوص حتى لو لم تذكر السؤال بنفس الكلمات.\\n-  إذا لم تكن النصوص تجيب على السؤال بشكل مباشر، قدم ما هو أقرب إجابة ممكنة من النصوص المتاحة وأوضح ذلك للمستخدم و لا تقل أبدًا لا توجد معلومات إلا إذا كانت النصوص لا علاقة لها بالسؤال تمامًا.\\n- لا تخترع معلومات غير موجودة في النصوص.\\n\\nالنصوص القانونية:\\n{context}\\n\\nالسؤال: {question} [/INST]"


def generate_answer(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda:0")
    input_length = inputs["input_ids"].shape[1]
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=True,
        temperature=0.1,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )
    new_tokens = outputs[0][input_length:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

st.title("⚖️ المساعد القانوني المصري")
st.caption("قانون العمل رقم 14 لسنة 2025 — الجريدة الرسمية")
st.divider()

question = st.text_area(
    "اكتب سؤالك القانوني:",
    height=100,
    placeholder="مثال: ما هي حقوقي إذا تم فصلي من العمل؟"
)

if st.button("ابحث في القانون ⚖️"):
    if not question.strip():
        st.warning("من فضلك اكتب سؤالاً.")
    else:
        with st.spinner("جاري البحث في نصوص القانون..."):
            results = retrieve(question, k=7)
            docs = [doc for doc, score in results]

        with st.spinner("جاري توليد الإجابة..."):
            prompt = build_prompt(question, docs)
            answer = generate_answer(prompt)

        st.success("الإجابة القانونية:")
        st.write(answer)
        st.divider()

        with st.expander("📖 المواد القانونية المستخدمة"):
            for doc, score in results:
                meta  = doc.metadata
                label = f"المادة {meta['article_label']}" if meta.get("article_label") else f"المادة ({meta['article']})"
                st.markdown(f"**{label}** — {meta.get('book', 'قانون الإصدار')} | Score: {score:.3f}")
                st.write(doc.page_content[:400] + "...")
                st.divider()

        st.caption("⚠️ هذا المساعد للاسترشاد فقط وليس استشارة قانونية رسمية.")
"""

with open('/kaggle/working/app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print("app.py updated.")

app.py updated.


In [15]:
!pip install pyngrok streamlit -q

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 83.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 102.7 MB/s eta 0:00:0000:010:01


In [16]:
import torch
import gc

del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()

print("GPU memory freed.")
print(f"GPU 0 free: {torch.cuda.mem_get_info(0)[0]/1024**3:.2f} GB")
print(f"GPU 1 free: {torch.cuda.mem_get_info(1)[0]/1024**3:.2f} GB")

GPU memory freed.
GPU 0 free: 14.41 GB
GPU 1 free: 14.41 GB


In [17]:
import subprocess, time
from pyngrok import ngrok

try:
    process.terminate()
except:
    pass
ngrok.kill()
time.sleep(3)

process = subprocess.Popen([
    "streamlit", "run", "/kaggle/working/app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
])

print("Waiting 4 minutes for models to load...")
time.sleep(240)

ngrok.set_auth_token("put token authentication here")
public_url = ngrok.connect(8501)
print(f"\n✅ Open this: {public_url}")

Waiting 4 minutes for models to load...




2026-07-30 13:17:42.146 Uvicorn server started on :::8501



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://34.21.71.149:8501

                                                                                                    
✅ Open this: NgrokTunnel: "https://dodgy-unbounded-cheating.ngrok-free.dev" -> "http://localhost:8501"


In [20]:
# run for restart
process.terminate()
ngrok.kill()
print("Stopped. Now re-run Cell above.")

Stopped. Now re-run Cell above.
